In [9]:
!pip install matplotlib numpy ipywidgets

In [10]:
# Kód pro de Casteljauův algoritmus a rasterizaci Bézierovy křivky

def lerp(A, B, t):
    """Lineární interpolace mezi dvěma body A a B pro parametr t v <0, 1>."""
    return (
        A[0] + t * (B[0] - A[0]),
        A[1] + t * (B[1] - A[1])
    )

def rasterizuj_usecku(x0, y0, x1, y1):
    """Pomocná funkce (DDA) pro převod jednoho rovného segmentu křivky na pixely."""
    dx = x1 - x0
    dy = y1 - y0
    kroky = max(abs(dx), abs(dy))
    if kroky == 0:
        return [(x0, y0)]
    
    x_inc = dx / kroky
    y_inc = dy / kroky
    x, y = float(x0), float(y0)
    pixely = []
    for _ in range(kroky + 1):
        pixely.append((round(x), round(y)))
        x += x_inc
        y += y_inc
    return pixely

def bezier_de_casteljau_didactic(P0, P1, P2, pocet_segmentu=5):
    """
    Kvadratická Bézierova křivka (3 řídicí body P0, P1, P2) pomocí de Casteljauova algoritmu.
    V každém kroku vypočítá bod B(t) pro dané t a rasterizuje úsečku z předchozího bodu.
    """
    steps = []
    vsechny_pixely = []
    uzly_krivky = []

    for k in range(pocet_segmentu + 1):
        t = round(k / pocet_segmentu, 2)

        # 1. úroveň interpolace (body Q0 a Q1 na řídicích ramenech)
        Q0 = lerp(P0, P1, t)
        Q1 = lerp(P1, P2, t)

        # 2. úroveň interpolace (výsledný bod B na Bézierově křivce)
        B = lerp(Q0, Q1, t)
        B_pixel = (round(B[0]), round(B[1]))
        uzly_krivky.append(B)

        # Rasterizace úsečky mezi předchozím a aktuálním bodem na křivce
        if k == 0:
            nove_pixely = [B_pixel]
        else:
            predchozi_B = uzly_krivky[k - 1]
            nove_pixely = rasterizuj_usecku(
                round(predchozi_B[0]), round(predchozi_B[1]),
                B_pixel[0], B_pixel[1]
            )

        for px in nove_pixely:
            if px not in vsechny_pixely:
                vsechny_pixely.append(px)

        steps.append({
            "krok": k,
            "t": t,
            "Q0": Q0,
            "Q1": Q1,
            "B_presne": B,
            "B_pixel": B_pixel,
            "nove_pixely": list(nove_pixely),
            "vykreslene_pixely": list(vsechny_pixely),
            "uzly_krivky": list(uzly_krivky)
        })

    return steps

In [11]:
# Kód pro vykreslení algoritmu do grafu

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

# Pevně dané řídicí body kvadratické Bézierovy křivky
P0 = (1, 1)
P1 = (5, 9)
P2 = (11, 2)
POCET_SEGMENTU = 5

historie = bezier_de_casteljau_didactic(P0, P1, P2, POCET_SEGMENTU)
max_krok = len(historie) - 1

aktualni_krok = 0

# Vytvoření ovládacích prvků
btn_zpet = widgets.Button(description='<- Krok zpět', disabled=True, button_style='info')
btn_vpred = widgets.Button(description='Krok vpřed ->', disabled=False, button_style='info')
lbl_stav = widgets.Label(value=f"Krok: 0 / {max_krok}")
vystup_grafu = widgets.Output(layout=widgets.Layout(min_height='650px'))

def vykresli_aktualni_stav():
    data_kroku = historie[aktualni_krok]
    t = data_kroku["t"]
    Q0 = data_kroku["Q0"]
    Q1 = data_kroku["Q1"]
    B_presne = data_kroku["B_presne"]
    B_pixel = data_kroku["B_pixel"]
    nove_pixely = data_kroku["nove_pixely"]

    max_x = max(P0[0], P1[0], P2[0]) + 3
    max_y = max(P0[1], P1[1], P2[1]) + 2

    # Popis nad graf
    if aktualni_krok == 0:
        nadpis = (
            f"Krok 0 z {max_krok} (parametr t = {t:.2f}) | Začínáme v prvním řídicím bodě P0 = {P0}\n"
            f"1. interpolace: Q0 = [{Q0[0]:.1f}, {Q0[1]:.1f}], Q1 = [{Q1[0]:.1f}, {Q1[1]:.1f}]  →  "
            f"2. interpolace: Bod na křivce B(0.0) = {B_pixel}."
        )
    else:
        predchozi_pixel = historie[aktualni_krok - 1]["B_pixel"]
        nadpis = (
            f"Krok {aktualni_krok} z {max_krok} (parametr t = {t:.2f} → jsme v {int(t*100)} % délky každého ramene)\n"
            f"1. interpolace: Q0 = [{Q0[0]:.2f}, {Q0[1]:.2f}], Q1 = [{Q1[0]:.2f}, {Q1[1]:.2f}]  →  "
            f"2. interpolace: B({t:.2f}) = [{B_presne[0]:.2f}, {B_presne[1]:.2f}] (pixel {B_pixel}).\n"
            f"Rasterizace: Vykreslíme rovnou úsečku z předchozího bodu {predchozi_pixel} do nového bodu {B_pixel}."
        )

    with vystup_grafu:
        vystup_grafu.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(9.5, 6.2))

        # 1. Vykreslení již rasterizovaných pixelů
        for px, py in data_kroku["vykreslene_pixely"]:
            je_novy = (px, py) in nove_pixely
            barva = '#ff6b6b' if je_novy else '#4ecdc4'
            alfa = 0.85 if je_novy else 0.5

            rect = patches.Rectangle((px - 0.5, py - 0.5), 1, 1,
                                     linewidth=1.4, edgecolor='black',
                                     facecolor=barva, alpha=alfa, zorder=2)
            ax.add_patch(rect)

        # 2. Řídicí polygon P0 -> P1 -> P2
        ax.plot([P0[0], P1[0], P2[0]], [P0[1], P1[1], P2[1]],
                color='#6c757d', linestyle=':', linewidth=2, marker='o', markersize=7,
                label='Řídicí polygon (P0, P1, P2)', zorder=3)
        ax.annotate(f"P0 {P0}", P0, textcoords="offset points", xytext=(-10, -18), fontweight='bold', color='#495057')
        ax.annotate(f"P1 {P1}", P1, textcoords="offset points", xytext=(-15, 8), fontweight='bold', color='#495057')
        ax.annotate(f"P2 {P2}", P2, textcoords="offset points", xytext=(8, -5), fontweight='bold', color='#495057')

        # 3. Ideální hladká Bézierova křivka
        t_hladke = np.linspace(0, 1, 100)
        bx = (1 - t_hladke)**2 * P0[0] + 2 * (1 - t_hladke) * t_hladke * P1[0] + t_hladke**2 * P2[0]
        by = (1 - t_hladke)**2 * P0[1] + 2 * (1 - t_hladke) * t_hladke * P1[1] + t_hladke**2 * P2[1]
        ax.plot(bx, by, color='#1a535c', linestyle='--', linewidth=2, label='Ideální Bézierova křivka', zorder=3)

        # 4. De Casteljauova pomocná úsečka Q0 -> Q1
        ax.plot([Q0[0], Q1[0]], [Q0[1], Q1[1]],
                color='#8338ec', linestyle='-', linewidth=2.2, marker='o', markersize=7,
                label=f'1. interpolace: úsečka Q0–Q1 (t={t:.2f})', zorder=4)

        # 5. Přesný bod B(t) na křivce
        ax.scatter([B_presne[0]], [B_presne[1]], color='#ffd166', edgecolor='black', s=100, zorder=6,
                   label=f'2. interpolace: bod na křivce B({t:.2f})')

        # Ukázka v legendě pro nově přidané pixely segmentu
        ax.scatter([], [], color='#ff6b6b', edgecolor='black', marker='s', s=80, label='Pixely aktuálního úseku křivky')

        # Nastavení mřížky
        ax.set_xticks(range(0, max_x))
        ax.set_yticks(range(0, max_y))
        ax.set_xticks([x - 0.5 for x in range(0, max_x + 1)], minor=True)
        ax.set_yticks([y - 0.5 for y in range(0, max_y + 1)], minor=True)

        ax.grid(which='minor', color='#cccccc', linestyle='-', linewidth=1)
        ax.grid(which='major', color='#eeeeee', linestyle=':', linewidth=0.5)

        ax.set_xlim(-0.5, max_x - 0.5)
        ax.set_ylim(-0.5, max_y - 0.5)
        ax.set_aspect('equal')

        # Panel nad grafem
        ax.set_title(
            nadpis,
            fontsize=10, pad=14, loc='left',
            bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='#dee2e6')
        )

        ax.legend(loc='upper right', fontsize=9)
        plt.tight_layout()
        plt.show()

# Obsluha kliknutí na tlačítka
def pri_kliknuti_zpet(b):
    global aktualni_krok
    if aktualni_krok > 0:
        aktualni_krok -= 1
        aktualizuj_ui()

def pri_kliknuti_vpred(b):
    global aktualni_krok
    if aktualni_krok < max_krok:
        aktualni_krok += 1
        aktualizuj_ui()

def aktualizuj_ui():
    btn_zpet.disabled = (aktualni_krok == 0)
    btn_vpred.disabled = (aktualni_krok == max_krok)
    lbl_stav.value = f"Krok: {aktualni_krok} / {max_krok}"
    vykresli_aktualni_stav()

btn_zpet.on_click(pri_kliknuti_zpet)
btn_vpred.on_click(pri_kliknuti_vpred)

# Složení panelu a vykreslení
ovladaci_panel = widgets.HBox([btn_zpet, btn_vpred, lbl_stav])
#display(vystup_grafu, ovladaci_panel)
#vykresli_aktualni_stav()

## Princip a pseudokód de Casteljauova algoritmu (Bézierova křivka)

**De Casteljauův algoritmus** slouží k výpočtu bodů na Bézierově křivce pomocí opakované **lineární interpolace** mezi řídicími body pro parametr $t \in \langle 0, 1 \rangle$. Samotná rasterizace do pixelové mřížky probíhá tak, že se parametr $t$ postupně zvyšuje po pevném kroku (např. $\Delta t = 0{,}2$) a mezi sousedními vypočtenými body na křivce se vykreslí krátké rovné úsečky (tzv. *aproximace lomenou čarou*).

Pro kvadratickou Bézierovu křivku danou třemi řídicími body $P_0, P_1, P_2$ probíhá výpočet ve dvou úrovních:
1. **1. interpolace:** Na úsečkách $P_0 P_1$ a $P_1 P_2$ najdeme body $Q_0(t)$ a $Q_1(t)$:
   $$Q_0(t) = (1 - t)P_0 + t P_1, \quad Q_1(t) = (1 - t)P_1 + t P_2$$
2. **2. interpolace:** Na spojnici $Q_0 Q_1$ najdeme výsledný bod $B(t)$ ležící přímo na křivce:
   $$B(t) = (1 - t)Q_0(t) + t Q_1(t)$$

```text
ALGORITMUS: Rasterizace Bézierovy křivky pomocí de Casteljauova algoritmu
VSTUP:  Řídicí body P0, P1, P2, počet úseků N
VÝSTUP: Vykreslené pixely křivky

1. Nastav výchozí bod křivky na první řídicí bod:
   predchozi_B = P0

2. Pro k od 1 do N opakuj:
   a) Urči hodnotu parametru t v intervalu <0, 1>:
      t = k / N
      
   b) 1. úroveň lineární interpolace (body na řídicím polygonu):
      Q0 = P0 + t * (P1 - P0)
      Q1 = P1 + t * (P2 - P1)
      
   c) 2. úroveň lineární interpolace (výsledný bod na křivce):
      B = Q0 + t * (Q1 - Q0)
      
   d) Vykresli rovnou úsečku z předchozího bodu do nového bodu:
      Vykresli_Usecku(zaokrouhli(predchozi_B), zaokrouhli(B))
      predchozi_B = B
```

In [12]:
display(vystup_grafu, ovladaci_panel)
vykresli_aktualni_stav()

Output(layout=Layout(min_height='650px'))